# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.
### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport pprint# Define the dataset URLurl = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(url)metadata = dataset.metadataprint(f"{metadata.name}: {metadata.description}")print(f"Version: {metadata.version}")print(f"Released: {getattr(metadata, 'datePublished', 'N/A')}")print("\nCite as:")print(metadata.citeAs)print("\nKeywords:")if hasattr(metadata, 'keywords'):    print(metadata.keywords)# Show a pretty-printed summary of metadata fieldsprint("\nAvailable top-level metadata attributes:")print(sorted(dir(metadata)))

## 2. Data OverviewReview available record sets, fields, and their IDs.We inspect the record sets, their fields, and columns via their `@id`s. This is essential for later referencing within `mlcroissant`.

In [ ]:
# List all record sets with their @idprint("Available Record Sets:")record_set_objs = []if hasattr(metadata, 'recordSet') and metadata.recordSet:    for rs in metadata.recordSet:        print(f"  @id: {rs['@id']}  name: {rs.get('name', 'N/A')}")        record_set_objs.append(rs)else:    # If the recordSet is not directly on Dataset, try loading from files/distributions    if hasattr(metadata, 'distribution') and metadata.distribution:        for dist in metadata.distribution:            # Try to access record sets from each distribution            if hasattr(dist, 'recordSet') and dist.recordSet:                for rs in dist.recordSet:                    print(f"  @id: {rs['@id']}  name: {rs.get('name', 'N/A')}")                    record_set_objs.append(rs)    else:        print("No accessible record sets discovered in metadata.")if not record_set_objs:    # As a fallback, try the experimental .record_sets attribute on dataset object    try:        for rs in dataset.record_sets:            print(f"  @id: {rs['@id']}  name: {getattr(rs, 'name', 'N/A')}")            record_set_objs.append(rs)    except Exception:        passprint(f"\nFound {len(record_set_objs)} record sets.")# For this dataset, get the list of record set idsrecord_set_ids = []for rs in record_set_objs:    # '@id' could be a property or key    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)    record_set_ids.append(rs_id)# List fields/columns for the first record set as exampleif record_set_objs:    print("\nFields/Columns of the first Record Set:")    rs0 = record_set_objs[0]    if isinstance(rs0, dict):        if 'field' in rs0:            for field in rs0['field']:                print(f"  Field @id: {field['@id']}  name: {field.get('name', 'N/A')}")        elif 'column' in rs0:            for col in rs0['column']:                print(f"  Column @id: {col['@id']}  name: {col.get('name', 'N/A')}")    else:        if hasattr(rs0, 'field') and rs0.field:            for field in rs0.field:                field_id = field['@id'] if isinstance(field, dict) else getattr(field, '@id', None)                name = field.get('name', '') if isinstance(field, dict) else getattr(field, 'name', 'N/A')                print(f"  Field @id: {field_id}  name: {name}")        elif hasattr(rs0, 'column') and rs0.column:            for col in rs0.column:                col_id = col['@id'] if isinstance(col, dict) else getattr(col, '@id', None)                name = col.get('name', '') if isinstance(col, dict) else getattr(col, 'name', 'N/A')                print(f"  Column @id: {col_id}  name: {name}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.For this analysis, we will use the first available record set.

In [ ]:
# If no record set is found, fall back to the default file object or fail gracefullyif not record_set_ids:    raise RuntimeError("No record set IDs could be found in the metadata.")# Use the first record set for demonstrationselected_record_set_id = record_set_ids[0]print(f"Selected record set @id: {selected_record_set_id}")dataframes = {}records = list(dataset.records(record_set=selected_record_set_id))df = pd.DataFrame(records)dataframes[selected_record_set_id] = dfprint("\nColumns in this DataFrame:")print(df.columns.tolist())df.head()

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.We'll search for a numeric field in the columns for analysis.

In [ ]:
# Try to heuristically select a numeric field for EDAnumeric_candidates = [c for c in df.columns if (df[c].dtype.kind in 'fi') and (df[c].nunique() > 2)]if not numeric_candidates:    # Try to select numeric by column name keywords    for c in df.columns:        if any(x in c.lower() for x in ['mw', 'weight', 'abundance', 'count', 'number', 'coverage', 'peptide']):            try:                df[c] = pd.to_numeric(df[c], errors='coerce')                if df[c].notnull().sum() > 0:                    numeric_candidates.append(c)            except Exception:                continue            # Pick the first candidate for analysisif numeric_candidates:    numeric_field = numeric_candidates[0]else:    raise RuntimeError("No suitable numeric field found for EDA.")print(f"Analyzing numeric field: {numeric_field}")# Thresholdthreshold = df[numeric_field].mean()filtered_df = df[df[numeric_field] > threshold].copy()print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")print(filtered_df.head())# Normalizefiltered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()print(f"\nNormalized {numeric_field} for filtered records:")print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())# Group by a possible categorical fieldcat_candidates = [c for c in df.columns if df[c].dtype == 'O' and df[c].nunique() < min(10, len(df)//5)]if cat_candidates:    group_field = cat_candidates[0]    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")    print(grouped_df.head())else:    print("No suitable categorical field found for grouping.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Visualization: Histogram of the selected numeric fieldplt.figure(figsize=(8, 5))sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)plt.title(f"Distribution of {numeric_field}")plt.xlabel(numeric_field)plt.ylabel("Frequency")plt.show()# If a categorical field exists, boxplot by groupif 'group_field' in locals():    plt.figure(figsize=(12, 5))    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)    plt.title(f"{numeric_field} by {group_field}")    plt.show()

## 6. ConclusionSummarize key findings and observations from the dataset exploration.This notebook demonstrated loading and analyzing the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` library.- Dataset metadata is easily accessible via the `.metadata` property.- Record sets and their fields can be referenced by `@id` for robust programmatic access.- The dataset contains at least one main record set with numerous numeric and categorical fields suitable for scientific analysis.- Exploratory steps included filtering, normalization, grouping, and visualization.For further analysis, consult the full dataset schema and documentation.